# 🚀 Day 16 – Project Hardening & Cleanup

Up until now we've been acting like ML engineers building features.

Today we'll act like backend engineers preparing a service for production.

The goal is not adding new recommendation logic.

The goal is:

Make the project cleaner
Make the project safer
Make the project easier to maintain


---

What we'll do today

Step 1

Move hardcoded values to environment variables

Current:

CACHE_TTL = 400

redis_client = redis.Redis(
    host="redis",
    port=6379
)

Problem:

Dev machine → maybe localhost
Docker → redis
Production → redis-prod.company.com

Need code change every time ❌


---

Step 2

Configuration Management

Instead of:

CACHE_TTL = 400
REDIS_HOST = "redis"
REDIS_PORT = 6379

Create:

config.py

Central place for settings.


---

Step 3

Pydantic Response Models

Currently:

return {
    "user_id": user_id,
    "recommendations": output
}

Works.

But FastAPI can't fully validate it.

We'll create schemas.


---

Step 4

Better Exception Handling

Current:

except Exception as e:
    return {"status":"FAILED"}

Too generic.

We'll make failures cleaner.


---

Step 5

Cleanup

Remove:

print(...)
debug statements
unused variables
duplicate endpoints


---

First Concept

Why Environment Variables?

Imagine:

Development

Redis Host = localhost

Docker

Redis Host = redis

Production

Redis Host = redis-production.company.com

Without environment variables:

redis_client = redis.Redis(
    host="localhost"
)

You must edit code.

Bad.


---

With environment variables:

import os

REDIS_HOST = os.getenv("REDIS_HOST")

Code never changes.

Only configuration changes.


---

Real Companies

Netflix

Amazon

Google

Almost everything important is configurable through:

Environment Variables
Config Files
Secrets Managers

NOT through source code.


---

Your First Task

Create a new file:

api/config.py

and put:

import os

REDIS_HOST = os.getenv("REDIS_HOST", "localhost")

REDIS_PORT = int(
    os.getenv("REDIS_PORT", 6379)
)

CACHE_TTL = int(
    os.getenv("CACHE_TTL", 400)
)


---

Question before we continue

Suppose Docker Compose contains:

environment:
  REDIS_HOST: redis

and your laptop contains nothing.

What value will:

os.getenv("REDIS_HOST", "localhost")

return

1. localhost


2. redis



And why?

Answer that first. This concept becomes extremely important in production systems. 😎

Good guess, but the answer is actually:

It returns:

"redis"

Why?

Because:

os.getenv("REDIS_HOST", "localhost")

means:

1. Check if REDIS_HOST exists in environment variables.
2. If yes → return that value.
3. If no → use "localhost" as default.

Think of it like:

if "REDIS_HOST" exists:
    return value_of_REDIS_HOST
else:
    return "localhost"


---

Example 1: Local Laptop

You don't set anything.

os.getenv("REDIS_HOST", "localhost")

Python checks:

REDIS_HOST exists? ❌

Returns:

"localhost"


---

Example 2: Docker Compose

environment:
  REDIS_HOST: redis

Now inside container:

REDIS_HOST exists? ✅
Value = redis

Returns:

"redis"


---

Why companies love this

Same code:

redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT
)

can run in:

Developer Laptop
Docker
QA Environment
Production

without changing a single line of code.

Only the environment variables change.


---

Quick Analogy

Imagine:

os.getenv("NAME", "Mugundhan")

Case 1:

NAME not set

Result:

"Mugundhan"

Case 2:

NAME=Netflix

Result:

"Netflix"

The second parameter is only a backup value.


---

Your task now

Create:

api/config.py

import os

REDIS_HOST = os.getenv("REDIS_HOST", "localhost")

REDIS_PORT = int(
    os.getenv("REDIS_PORT", 6379)
)

CACHE_TTL = int(
    os.getenv("CACHE_TTL", 400)
)

Then tell me once it's done.

Next we'll replace the hardcoded values in app.py and wire them into Docker Compose like a production application. 🚀

Perfect. 🚀

Now we're doing something that separates a learning project from a production-style project.


---

Step 2: Use Config Values in app.py

Currently you have:

CACHE_TTL = 400

redis_client = redis.Redis(
    host="redis",
    port=6379,
    decode_responses=True
)

We want:

app.py
    ↓
config.py
    ↓
Environment Variables


---

Add Import

Near the top of app.py:

from api.config import (
    REDIS_HOST,
    REDIS_PORT,
    CACHE_TTL
)


---

Remove

Delete:

CACHE_TTL = 400


---

Replace Redis Connection

Replace:

redis_client = redis.Redis(
    host="redis",
    port=6379,
    decode_responses=True
)

with:

redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    decode_responses=True
)


---

Why is this better?

Before:

app.py knows everything

After:

app.py
 ↓
config.py
 ↓
environment variables

If tomorrow:

Redis Host changes
Redis Port changes
Cache TTL changes

You don't touch application code.


---

Step 3: Add Environment Variables to Docker Compose

Currently:

api:
  build: .
  container_name: recommendation-api

  ports:
    - "8001:8001"

  depends_on:
    - redis

Change to:

api:
  build: .
  container_name: recommendation-api

  ports:
    - "8001:8001"

  environment:
    REDIS_HOST: redis
    REDIS_PORT: 6379
    CACHE_TTL: 400

  depends_on:
    - redis


---

Concept Check

Now when container starts:

Docker injects:

REDIS_HOST=redis
REDIS_PORT=6379
CACHE_TTL=400

into the API container.

Then:

REDIS_HOST = os.getenv("REDIS_HOST", "localhost")

becomes:

REDIS_HOST = "redis"

automatically.


---

Verify

Add temporarily in startup:

print("REDIS_HOST =", REDIS_HOST)
print("REDIS_PORT =", REDIS_PORT)
print("CACHE_TTL =", CACHE_TTL)

inside:

def load_resources():

Just for learning.

Expected:

REDIS_HOST = redis
REDIS_PORT = 6379
CACHE_TTL = 400

when running through Compose.


---

After changes

Run:

docker compose down

Then:

docker compose up --build

Why build again?

Because:

app.py changed
config.py added
docker-compose.yml changed

So we want a fresh image.


---

Mini Homework Question

Suppose tomorrow production wants:

Cache TTL = 1800 seconds

With our new setup:

Do we change:

A)

CACHE_TTL = 1800

inside code

or

B)

CACHE_TTL: 1800

inside configuration

And why?

Answer that after you've run the rebuild and verified the printed values. 🎯

Exactly right. 🎯

That's the mindset production engineers use.

Before:

CACHE_TTL = 400

If Product Team says:

> "Increase cache duration to 1800 seconds"



You would need:

Edit code
↓
Commit
↓
Push
↓
Rebuild image
↓
Redeploy


---

Now:

environment:
  CACHE_TTL: 1800

Only configuration changes.

Application code remains:

CACHE_TTL = int(
    os.getenv("CACHE_TTL", 400)
)

unchanged.


---

Real-world analogy

Think of:

os.getenv(...)

as a TV.

Think of:

environment:

as the remote control.

You don't open the TV and change its circuits every time you want a different channel.

You change the remote settings.


---

One subtle concept

Notice we still have:

os.getenv("CACHE_TTL", 400)

Why keep 400?

Suppose a developer runs:

uvicorn api.app:app --reload

without Docker.

No environment variable exists.

Then:

CACHE_TTL = 400

automatically.

So:

Environment Variable exists?
      |
      ├── Yes → Use it
      |
      └── No → Use default

This makes the application portable.


---

Day 16 Progress

✅ Configuration extracted

✅ Environment variables introduced

✅ Docker Compose configuration driven

You've now crossed another production-level concept.


---

Next Hardening Step: Pydantic Response Models

Right now:

return {
    "user_id": user_id,
    "recommendations": output
}

FastAPI accepts it.

But it doesn't know the exact structure beforehand.

We'll create schemas like:

class Recommendation(BaseModel):
    movie: str
    score: float

and

class RecommendationResponse(BaseModel):
    user_id: int
    recommendation_count: int
    recommendations: list[Recommendation]

Benefits:

✅ Automatic validation

✅ Cleaner code

✅ Better Swagger docs

✅ Industry standard FastAPI practice

Before we code it:

Quick question: If by mistake your code returns:

{
    "movie": 123,
    "score": "high"
}

instead of:

{
    "movie": "Titanic",
    "score": 0.98
}

would you prefer:

A) The API silently returns bad data

B) FastAPI catches the mistake immediately

Which one and why? 👀

Exactly. 🎯

That's one of the biggest reasons schemas exist.

Imagine tomorrow during a refactor you accidentally write:

output.append(
    {
        "movie": 123,
        "score": "high"
    }
)

Without validation:

API returns garbage
↓
Frontend receives garbage
↓
Bug appears somewhere else
↓
Debugging nightmare


---

With Pydantic:

class Recommendation(BaseModel):
    movie: str
    score: float

FastAPI immediately says:

Validation Error
movie should be string
score should be float

and you catch the bug at the source.


---

Step 4 - Create Schemas

Create a new file:

api/schemas.py

Add:

from pydantic import BaseModel


class Recommendation(BaseModel):

    movie : str

    score : float


class RecommendationResponse(BaseModel):

    model_version : str

    user_id : int

    recommendation_count : int

    recommendations : list[Recommendation]

    response_time : float


---

What does this represent?

Previously:

{
    "model_version": "svd_model_v1.pkl",
    "user_id": 1,
    "recommendation_count": 10,
    "recommendations": [...],
    "response_time": 0.02
}

was just a dictionary.

Now it becomes:

RecommendationResponse

which formally describes your API contract.

Think:

Frontend Team
      +
Backend Team

Both agree:

Response will look like THIS

That's why schemas are so important in large teams.


---

Step 5 - Import Schema

In app.py

Add:

from api.schemas import RecommendationResponse


---

Step 6 - Use response_model

Change:

@app.get("/recommend/{user_id}")

to:

@app.get(
    "/recommend/{user_id}",
    response_model = RecommendationResponse
)

That's it.

No other code changes needed yet.


---

What FastAPI now does

When endpoint returns:

response

FastAPI automatically checks:

model_version -> str ?
user_id -> int ?
recommendation_count -> int ?
recommendations -> list ?
response_time -> float ?

If not:

Validation Error

instead of silently returning bad data.


---

Magic You'll Notice

Open:

http://localhost:8001/docs

after rebuilding.

You'll now see a detailed schema for:

GET /recommend/{user_id}

Swagger knows exactly what the API returns.

This is why FastAPI became popular in the first place.


---

After adding schemas.py

Run:

docker compose down
docker compose up --build

Then:

localhost:8001/docs

and check if the recommendation endpoint now shows a structured response model.

Once that's working, we'll move to the next hardening step:

🛡️ Cleaning up error handling and making the API return proper HTTP status codes instead of just { "status": "FAILED" }.

That's another thing interviewers love to ask about FastAPI projects. 🚀

Perfect. 🚀

You've now implemented something many FastAPI tutorials skip:

✅ Configuration Management
✅ Environment Variables
✅ Docker Compose Configuration
✅ Pydantic Response Models

Your API is already looking much more professional.


---

Next Hardening Step: Proper HTTP Error Handling

Current code:

if user_data.empty:

    return {
        "status": "error",
        "message": "Invalid User name give range below 900"
    }

and

except Exception as e:

    return {
        "status": "FAILED",
        "error": str(e)
    }

Problem:

Even when an error occurs:

HTTP Status = 200 OK

The client sees:

{
  "status": "FAILED"
}

but HTTP still says:

200 OK

which is misleading.


---

Real-world Example

Imagine Netflix API:

GET /recommendations/999999

Invalid user.

Would Netflix return:

200 OK

?

No.

It would return:

404 Not Found

or

400 Bad Request


---

FastAPI Way

Import:

from fastapi import HTTPException


---

Replace Invalid User Logic

Current:

if user_data.empty:

    return {
        "status": "error",
        "message": "Invalid User name give range below 900"
    }

Replace with:

if user_data.empty:

    raise HTTPException(
        status_code = 404,
        detail = f"User {user_id} not found"
    )


---

What Happens Now?

User calls:

/recommend/999999

Response:

{
  "detail": "User 999999 not found"
}

HTTP Status:

404 Not Found

Much better.


---

Next

Current:

except Exception as e:

    return {
        "status": "FAILED",
        "error": str(e)
    }

Replace with:

except Exception as e:

    error_count += 1

    logging.error(str(e))

    raise HTTPException(
        status_code = 500,
        detail = "Internal Server Error"
    )

Notice:

Don't expose raw exception details

to users.

In production:

Bad:

{
  "error": "Redis connection timeout..."
}

Good:

{
  "detail": "Internal Server Error"
}

while logs contain the real error.


---

Why This Matters

Currently:

User sees:
Error details
Stack information
Internal implementation

Potential security issue.

Production systems usually:

User -> Generic Error

Logs -> Detailed Error


---

Interview Question

Suppose Redis crashes.

Should the client receive:

{
  "error": "ConnectionRefusedError: redis://..."
}

or

{
  "detail": "Internal Server Error"
}

and why?

Answer that first, then make the changes and test:

/recommend/999999

You should see your first proper:

404 Not Found

response. 🔥

You're very close. The issue is a classic FastAPI exception-handling mistake.

Look at this:

if user_data.empty:

    raise HTTPException(
        status_code = 404,
        detail = f"User {user_id} not found"
    )

This is correct.

But later you have:

except Exception as e:

    error_count += 1

    logging.error(str(e))

    return HTTPException(
        status_code = 500,
        detail = "Internal server Error"
    )

Problem 1

When you raise:

raise HTTPException(404, ...)

it is also an Exception.

So your code flow becomes:

raise HTTPException(404)
        ↓
except Exception catches it
        ↓
returns 500

Therefore your 404 is getting converted into a 500.


---

Problem 2

Inside the except block you should use:

raise HTTPException(...)

not

return HTTPException(...)

HTTPException is meant to be raised.


---

Correct Fix

Add this import:

from fastapi import HTTPException

(you already have it)

Then change your exception handling to:

except HTTPException:
    raise

followed by:

except Exception as e:

    error_count += 1

    logging.error(str(e))

    raise HTTPException(
        status_code = 500,
        detail = "Internal Server Error"
    )

So the full structure becomes:

try:

    ...

    if user_data.empty:

        raise HTTPException(
            status_code = 404,
            detail = f"User {user_id} not found"
        )

    ...

except HTTPException:
    raise

except Exception as e:

    error_count += 1

    logging.error(str(e))

    raise HTTPException(
        status_code = 500,
        detail = "Internal Server Error"
    )


---

Why this works

Invalid User
     ↓
HTTPException(404)
     ↓
except HTTPException
     ↓
raise
     ↓
404 returned to client

But:

Redis failure
     ↓
ConnectionError
     ↓
except Exception
     ↓
500 Internal Server Error


---

Small interview question

Why do we write:

except HTTPException:
    raise

instead of letting:

except Exception

handle everything?

Try answering that before implementing the fix. This is exactly the kind of reasoning interviewers look for when discussing API error handling. 🚀

You're on the right track, but there's an even more important reason.

Your answer:

> Because not every error is same and if it is related to http we can clearly debug later if necessary



✅ Partially correct.

The deeper reason is:

HTTPException is NOT actually an error in our application.
It is an intentional response.

Example:

raise HTTPException(
    status_code=404,
    detail="User not found"
)

This doesn't mean:

Something broke ❌

It means:

The API successfully handled the request
and decided the correct response is 404 ✅


---

If we don't do:

except HTTPException:
    raise

then:

raise HTTPException(404)

gets caught by:

except Exception

and becomes:

500 Internal Server Error

which tells the client:

Server is broken ❌

even though:

Server is perfectly fine
User simply doesn't exist ✅


---

Think of it this way:

User doesn't exist

Request:
GET /recommend/999999

Response:
404 Not Found

Normal business scenario.


---

Redis crashes

Request:
GET /recommend/1

Response:
500 Internal Server Error

Actual system failure.


---

Interviewers love this distinction:

4xx → Client-side problem
5xx → Server-side problem

Examples:

Status	Meaning

400	Bad request
401	Unauthorized
403	Forbidden
404	Not found
500	Internal server error
503	Service unavailable



---

One more improvement

Currently you have:

except Exception as e:

    error_count += 1

    logging.error(str(e))

    raise HTTPException(
        status_code=500,
        detail="Internal Server Error"
    )

This is good.

But now notice:

except HTTPException:
    raise

will prevent:

404 errors

from increasing:

error_count

which is actually correct.

Because:

User not found

is not a system error.

But:

Redis down
Model load failed
JSON parsing failed

are real system errors.

So your metrics become more meaningful.


---

After implementing the fix, test:

/recommend/1

Expected:

200 OK

and

/recommend/999999

Expected:

404 Not Found
{
  "detail": "User 999999 not found"
}

If both work, Day 16's error-handling section is complete, and we'll move to API cleanup and production-ready folder structure. 🚀

Awesome. 🚀

At this point your API is already much better than the average portfolio FastAPI project.

So let's do the next hardening phase.

Step 6: Remove Debug/Development Code

I can already spot some things in your app.py that were useful for learning but shouldn't be in the final version.

For example:

print ( "redis type", type ( redis_client ) )

print("HAS ATTRIBUTE", hasattr(redis_client, "scan_iter"))

These were temporary debugging checks.

Now that Redis is working:

❌ Remove them.


---

Also:

# model, model_name = load_latest_svd_model()

# similarity = load_latest_similarity()

These old commented lines should go.

Production code shouldn't contain dead code.


---

Step 7: Improve Logging

Currently:

logging.info(
    f" Response Time : { round ( end - start, 2 ) } sec "
)

Good.

Let's make logs more informative.

Replace with:

logging.info(
    f"User={user_id} | "
    f"Recommendations={n} | "
    f"ResponseTime={round(end-start,2)} sec"
)

Now logs become:

2026-06-04 19:20:10
INFO
User=25 | Recommendations=10 | ResponseTime=0.14 sec

Much easier to investigate later.


---

Step 8: Add API Metadata

Currently:

app = FastAPI()

Replace with:

app = FastAPI(

    title = "Movie Recommendation API",

    description =
    "Hybrid Recommendation System using SVD, Content-Based Filtering and Redis Cache",

    version = "1.0.0"
)


---

Now open:

localhost:8001/docs

You'll see:

Movie Recommendation API
Version 1.0.0

Looks much more professional.


---

Step 9: Tag Endpoints

Example:

@app.get(
    "/recommend/{user_id}",
    tags = ["Recommendations"],
    response_model = RecommendationResponse
)


---

@app.get(
    "/health",
    tags = ["Monitoring"]
)


---

@app.get(
    "/metrics",
    tags = ["Monitoring"]
)


---

Swagger UI becomes organized:

Recommendations
 └── /recommend

Monitoring
 ├── /health
 ├── /metrics
 ├── /cache-stats

Instead of a giant list.


---

Concept Check

Why do tags matter?

Imagine:

5 endpoints

No big deal.

But imagine:

200 endpoints

for:

Users
Movies
Recommendations
Admin
Monitoring
Authentication
Payments
Analytics

Without tags:

Huge mess

With tags:

Clean sections


---

Your Task

Implement these 4 improvements:

✅ Remove debug prints

✅ Improve log message

✅ Add FastAPI title/description/version

✅ Add tags to endpoints

Then rebuild:

docker compose down
docker compose up --build

Open:

http://localhost:8001/docs

and observe the difference.


---

Mini question while doing it:

Suppose tomorrow you add:

/login
/logout
/register

Would you put them under:

Recommendations

or create a new tag?

If new tag, what would you name it? 👀

Your thinking is actually moving toward authentication flow, which is good, but it answers a slightly different question.

I asked:

> If we add /login, /logout, /register, should they be under the Recommendations tag?



The answer would be:

No

We should create a new tag:

Authentication

So Swagger would look like:

Authentication
 ├── /login
 ├── /logout
 └── /register

Recommendations
 └── /recommend/{user_id}

Monitoring
 ├── /health
 ├── /metrics
 ├── /cache-stats
 └── /clear-cache


---

Now, regarding your thought:

> once it hits the fastapi port we need to redirect it to login/register page



That is actually how many web applications work.

Example:

User opens Netflix
        ↓
Authenticated?
        ↓
      No
        ↓
Login Page
        ↓
Login Success
        ↓
Home Page
        ↓
Recommendations API

But notice something:

Frontend Responsibility

Usually:

React
Angular
Vue
Mobile App

decides:

User logged in?

and redirects to Login Page.


---

Backend Responsibility

FastAPI usually does:

/login

and returns:

{
  "token": "abc123"
}

Then every future request:

GET /recommend/1

Authorization: Bearer abc123


---

So in a production recommendation system:

Frontend
    ↓
Login Page
    ↓
Calls FastAPI /login
    ↓
Receives Token
    ↓
Calls /recommend

The backend doesn't normally "show pages."

It provides APIs.


---

Architecture View

Your project currently:

User
 ↓
FastAPI
 ↓
Recommendation Engine
 ↓
Redis

A real production version might be:

User
 ↓
Frontend (React)
 ↓
Authentication API
 ↓
Recommendation API
 ↓
Redis
 ↓
ML Models


---

So your intuition is good because you're already thinking:

How does a real application work?

which is exactly the next level of thinking.


---

One question for you

Suppose we add authentication later.

Which is safer?

Option A

GET /recommend/1

User can type any ID.

Option B

GET /recommend

Backend reads the logged-in user's identity from the token and automatically knows:

Current User = 1

Which option do you think Netflix/Amazon would use and why? 🤔

Correct. ✅

Netflix, Amazon, YouTube, Spotify almost always use something closer to Option B.

Reason:

User logs in
      ↓
Backend identifies user from token
      ↓
Backend fetches recommendations for THAT user

Not:

GET /recommend/123

because someone could simply change:

123 → 124

and potentially access another user's data.

That's a concept called:

Authorization

We'll discuss it more when we talk about production architectures and system design.


---

Next Hardening Step: Folder Structure Cleanup

Right now your project is probably something like:

project/
│
├── api/
│   ├── app.py
│   ├── config.py
│   └── schemas.py
│
├── src/
│   ├── collaborative.py
│   ├── content_based.py
│   ├── hybrid.py
│   └── data_loader.py
│
├── models/
├── logs/
├── Dockerfile
├── docker-compose.yml
└── requirements.txt

That's already decent.


---

Why structure matters

Imagine after 2 years:

50 Python files
10 developers
200 endpoints

Without organization:

Where is recommendation logic?
Where are schemas?
Where is config?
Where are models?

Chaos.


---

A Common Production Layout

For a project of your size, something like:

project/

├── api/
│   ├── app.py
│   ├── config.py
│   └── schemas.py
│
├── src/
│   ├── collaborative.py
│   ├── content_based.py
│   ├── hybrid.py
│   └── data_loader.py
│
├── models/
│   ├── svd_model_v1.pkl
│   └── similarity_v1.pkl
│
├── logs/
│   └── api.log
│
├── data/
│   └── movies.csv
│
├── Dockerfile
├── docker-compose.yml
├── requirements.txt
└── README.md

Honestly, you're already very close to this.

So I would not spend time doing a major refactor right now.

The gain is small compared to what you've already built.


---

What I'd do instead

Let's make the API documentation cleaner.

Add summaries.

Example:

@app.get(
    "/recommend/{user_id}",
    tags=["Recommendations"],
    summary="Get movie recommendations for a user",
    response_model=RecommendationResponse
)


---

For health:

@app.get(
    "/health",
    tags=["Monitoring"],
    summary="Check API health"
)


---

For metrics:

@app.get(
    "/metrics",
    tags=["Monitoring"],
    summary="View API metrics"
)


---

Now Swagger becomes:

Recommendations
  Get movie recommendations for a user

Monitoring
  Check API health
  View API metrics

instead of generic endpoint names.


---

Why this matters

In companies:

Frontend Team
QA Team
DevOps Team
Data Team

may all use your API.

Good documentation saves a lot of time.


---

Your next task

Add:

summary=... to all endpoints

Keep the tags you added earlier


Then open:

http://localhost:8001/docs

and look at how much more polished it feels.

After that, we'll do one of the final Day 16 production topics:

🚀 Environment-specific configuration (Dev vs Docker vs Production)

which is a very common interview topic and a nice finishing touch before Day 17 (GitHub-ready project).


Perfect. 🚀

We're entering one of the most valuable production concepts now.

Environment-Specific Configuration

Right now we have:

REDIS_HOST = os.getenv("REDIS_HOST", "localhost")

REDIS_PORT = int(
    os.getenv("REDIS_PORT", 6379)
)

CACHE_TTL = int(
    os.getenv("CACHE_TTL", 400)
)

This is already good.

But real systems usually have multiple environments:

Development
Testing / QA
Production


---

Why?

Imagine:

Development

REDIS_HOST = localhost
CACHE_TTL = 60
DEBUG = True

You want fast testing.


---

QA

REDIS_HOST = qa-redis
CACHE_TTL = 300
DEBUG = False

Closer to production.


---

Production

REDIS_HOST = prod-redis
CACHE_TTL = 1800
DEBUG = False

Maximum performance.


---

Current Situation

Your project already supports this.

Because:

os.getenv(...)

reads environment variables.

You can change behavior without touching code.


---

Interview Question

Suppose Production Team says:

Increase CACHE_TTL to 3600

Which is safer?

Option A

CACHE_TTL = 3600

commit code


---

Option B

Update environment variable
Restart service

The answer is:

Option B

because:

No code changes
No new bugs introduced
No redeployment risk


---

Another Production Concept

Secrets

Currently:

REDIS_HOST
REDIS_PORT
CACHE_TTL

are fine.

But imagine:

DATABASE_PASSWORD
JWT_SECRET
API_KEYS

Would we store them in code?

DB_PASSWORD = "admin123"

❌ Never.

Instead:

DB_PASSWORD = os.getenv("DB_PASSWORD")

This is why environment variables are heavily used.


---

Project Review

Look how far we've come.

Early Days

Jupyter Notebook
↓
Train SVD
↓
Print Recommendations


---

Today

FastAPI
↓
Hybrid Recommendation Engine
↓
Pydantic Validation
↓
Redis Distributed Cache
↓
Docker
↓
Docker Compose
↓
Environment Variables
↓
Monitoring Endpoints
↓
Structured Logging
↓
HTTP Error Handling

This is a huge jump.


---

Day 16 Remaining Work

We're almost done.

The last cleanup item I'd recommend is:

Remove startup deprecation warning

You're currently using:

@app.on_event("startup")

Modern FastAPI prefers:

lifespan()

The old one still works, but produces deprecation warnings in newer versions.


---

Quick Concept Question

Current:

@app.on_event("startup")

def load_resources():

    app.state.model = ...

What is the purpose of loading models during startup instead of inside:

def recommend_movies(...)

Why don't we simply load the SVD model every time a recommendation request arrives?

Think about performance, memory, and response time. This is another very common production-system question. 🔥

Your answer about model loading is partially correct, but there's one bigger reason.

If we loaded the model inside every request:

@app.get("/recommend/{user_id}")
def recommend_movies():

    model = load_latest_svd_model()

    ...

then for every API call:

User 1 request
↓
Load SVD model from disk
↓
Generate recommendation

User 2 request
↓
Load SVD model from disk
↓
Generate recommendation

User 3 request
↓
Load SVD model from disk
↓
Generate recommendation

That would be extremely slow.

The production pattern is:

Server Starts
↓
Load model once into memory
↓
Keep model in RAM
↓
Thousands of requests reuse same model

So the main reason is:

Load once
Reuse many times

rather than:

Load every request


---

Now your question:

> For prod, qa, dev do we have separate separate env variables right?



✅ Exactly.

Typically you have different environment configurations.

Example:

Development

.env.dev

REDIS_HOST=localhost
REDIS_PORT=6379
CACHE_TTL=60
DEBUG=True


---

QA

.env.qa

REDIS_HOST=qa-redis
REDIS_PORT=6379
CACHE_TTL=300
DEBUG=False


---

Production

.env.prod

REDIS_HOST=prod-redis
REDIS_PORT=6379
CACHE_TTL=1800
DEBUG=False

Same codebase.

Different configuration.


---

Think of it like this:

You = Same person

But:

At home
↓
Casual clothes

At office
↓
Formal clothes

At wedding
↓
Suit

You didn't change yourself.

Only the environment changed.

Similarly:

app.py

stays the same.

Only:

.env.dev
.env.qa
.env.prod

changes.

This is one of the core ideas behind the famous:

"Twelve-Factor App"

principle used in modern cloud-native systems.


---

Where are we now?

You've completed:

✅ FastAPI
✅ Hybrid Recommender
✅ Model Versioning
✅ Redis Distributed Cache
✅ Cache Invalidation Strategies
✅ Docker
✅ Docker Compose
✅ Environment Variables
✅ Pydantic Schemas
✅ HTTP Error Handling
✅ Monitoring Endpoints

At this point, your project is already stronger than many entry-level ML portfolio projects.


---

Next Concept Before Day 17

I would cover:

Lifespan Events (Modern FastAPI)

because it directly replaces:

@app.on_event("startup")

which FastAPI is gradually moving away from.

It's not a huge code change, but it teaches an important production concept:

Application Startup
↓
Resource Initialization
↓
Application Shutdown
↓
Resource Cleanup

and after that we'll be ready to make the project GitHub-ready with README, architecture diagram, and resume points.

So yes, I'd move to Lifespan Events next. 🚀

Perfect. This is exactly how you should think about these changes:

> Not just "what code changes?", but "what architectural problem does it solve?"




---

Current Way (Startup Event)

Today you have:

@app.on_event("startup")
def load_resources():

    app.state.model, app.state.model_name = load_latest_svd_model()

    app.state.similarity = load_latest_similarity()

    print("Resource loaded...")


---

Flow

FastAPI Starts
        │
        ▼
Startup Event Triggered
        │
        ▼
Load SVD Model
        │
        ▼
Load Similarity Matrix
        │
        ▼
Store in app.state
        │
        ▼
API Ready


---

Request Flow

User Request
       │
       ▼
recommend()
       │
       ▼
Uses app.state.model
       │
       ▼
Recommendation

Good.


---

Problem with Startup Events

FastAPI team is moving towards a newer lifecycle model.

Reason:

Startup only handles:

Start

What about:

Shutdown?

For example:

Database Connection
Redis Connection
Message Queue
Kafka Consumer
GPU Resources
File Handles

Should be cleaned when app exits.

Startup event doesn't naturally pair:

Initialize
Cleanup

into one flow.


---

New Lifespan Pattern

Instead of:

@app.on_event("startup")

we use:

from contextlib import asynccontextmanager


---

Example:

from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app: FastAPI):

    print("Loading resources...")

    app.state.model, app.state.model_name = load_latest_svd_model()

    app.state.similarity = load_latest_similarity()

    yield

    print("Cleaning resources...")


---

Then:

app = FastAPI(
    lifespan=lifespan
)


---

New Flow

FastAPI Starts
        │
        ▼
lifespan()
        │
        ▼
Load Model
        │
        ▼
Load Similarity
        │
        ▼
yield
        │
        ▼
Application Running


---

When server stops:

CTRL + C
       │
       ▼
After yield executes
       │
       ▼
Cleanup Logic
       │
       ▼
App Shutdown


---

Visual Comparison

Startup Event

START
  │
  ▼
Load Resources
  │
  ▼
Run App

No shutdown handling.


---

Lifespan

START
  │
  ▼
Load Resources
  │
  ▼
yield
  │
  ▼
Run App
  │
  ▼
Cleanup Resources
  │
  ▼
STOP

Complete lifecycle.


---

Real Production Example

Imagine Netflix API.

Startup:

Connect Redis
Connect DB
Load ML Models
Create Kafka Consumer


---

Shutdown:

Close Redis
Close DB
Close Kafka Consumer
Release GPU Memory


---

Lifespan allows:

Initialize Everything
Run Application
Cleanup Everything

inside one place.

That's why modern FastAPI prefers it.


---

For Our Project

Today:

Load SVD Model
Load Similarity Matrix

No cleanup required.

So:

Startup Event ✅ Works
Lifespan ✅ Better Design

Functionally identical right now.

But lifespan teaches the pattern used in larger systems.


---

Interview Angle

Suppose interviewer asks:

> Why load the model during application startup instead of inside the endpoint?



Strong answer:

Loading the model on every request would cause repeated disk I/O,
increase latency, and waste CPU resources.

Loading once during startup keeps the model in memory and allows all requests
to reuse the same instance, significantly improving response time and throughput.


---

Small Check

If we later add:

Redis Connection
Database Connection
Kafka Consumer
SVD Model
Similarity Matrix

Where would you prefer keeping all initialization logic?

Option A

@app.on_event("startup")
def load_model():

@app.on_event("startup")
def load_db():

@app.on_event("startup")
def load_redis():

@app.on_event("startup")
def load_kafka():

Option B

One centralized:

lifespan()

containing:

Initialize Everything
↓
yield
↓
Cleanup Everything

Which one looks easier to maintain and why? 👀

Yes. ✅

I would officially consider Day 16 completed.

Look at where the project stands now:

Day 1

Load MovieLens
↓
Simple recommendations
↓
Notebook

Day 16

MovieLens
↓
Hybrid Recommendation Engine
↓
SVD + Content Based
↓
FastAPI
↓
Response Models
↓
Error Handling
↓
Monitoring Endpoints
↓
Structured Logging
↓
Redis Distributed Cache
↓
Cache Invalidation Strategies
↓
Docker
↓
Docker Compose
↓
Environment Variables
↓
Production-style Configuration
↓
API Documentation
↓
Lifespan Concepts

That's a massive jump. 🔥


---

Revised Roadmap

Day 17 → GitHub Ready Project

Goal:

Make project look like something
a recruiter/interviewer can clone and run.

We'll do:

README.md

Professional README containing:

Project Overview
Architecture
Features
Tech Stack
Installation
Docker Setup
API Endpoints
Sample Output
Future Improvements


---

Architecture Diagram

Something like:

User
 │
 ▼
FastAPI
 │
 ├── Redis Cache
 │
 ├── SVD Model
 │
 └── Content-Based Engine
 │
 ▼
Recommendations


---

Clean Folder Structure Review

Final cleanup before GitHub.


---

.gitignore

Avoid pushing:

__pycache__
logs
venv
.ipynb_checkpoints

and other junk.


---

Day 18 → Performance & Production Review

This is where we think like a Senior Engineer.

Topics:

If traffic increases 100x

1 User
↓
100 Users
↓
10,000 Users

What breaks first?


---

Load Balancing

User
 ↓
Load Balancer
 ↓      ↓      ↓
API1   API2   API3


---

Why Redis becomes important

Without Redis:

Each API calculates recommendations

With Redis:

One calculation
↓
Many cache hits


---

Bottleneck Analysis

We'll inspect:

CPU
RAM
Disk I/O
Network
Redis
Model Loading


---

Day 19 → Resume + Interview Preparation

This is where we convert project work into:

Resume Bullet

Something like:

Built a production-style hybrid recommendation system
using collaborative filtering (SVD), content-based filtering,
FastAPI, Redis caching, Docker and Docker Compose,
reducing recommendation latency through distributed caching.


---

Interview Questions

You'll be able to answer:

Why Redis?
Why Docker?
Why FastAPI?
Why SVD?
Why Hybrid?
How cache invalidation works?
How scaling works?


---

Day 20 → Final System Design Review

We'll do a complete walkthrough:

Netflix-style Recommendation System

and connect everything you've learned.


---

After Day 20

You'll have a project containing:

✅ ML
✅ Recommendation Systems
✅ APIs
✅ Docker
✅ Redis
✅ Monitoring
✅ Production Concepts
✅ Deployment Knowledge
✅ System Design Foundations

This is exactly the type of project that starts moving someone from:

"Knows Machine Learning"

toward

"Can build ML Systems"

which is the direction AI/ML Engineer roles expect.

So next session we'll begin:

Day 17 — GitHub Ready Project 🚀

and turn this from a learning project into a portfolio project.